# 04 ? Evaluation and export

Summarize notebook 03 metrics, compare test forecasts, inspect residuals for the best available tree model, record the experiment, and report dashboard artifact readiness.

In [ ]:
## Setup

import os

import sys, warnings, yaml, joblib
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'notebook'

_cwd = Path.cwd().resolve()
_candidates = [_cwd, *_cwd.parents]
_candidates.extend(path.parent for path in _cwd.glob("*/configs/pipeline.yaml"))
PROJECT_ROOT = next((path for path in _candidates if (path / "configs" / "pipeline.yaml").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find configs/pipeline.yaml. Open the project folder or set the notebook working directory to this repository.")
os.chdir(PROJECT_ROOT)
CONFIG_PATH = PROJECT_ROOT / "configs" / "pipeline.yaml"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.experiment_tracker import log_run

with open(PROJECT_ROOT / 'configs' / 'pipeline.yaml') as f:
    CFG = yaml.safe_load(f)

PROCESSED = PROJECT_ROOT / CFG['artifacts']['processed_dir']
MODELS    = PROJECT_ROOT / CFG['artifacts']['models_dir']
FIGURES   = PROJECT_ROOT / CFG['reports']['figures_dir']
METRICS   = PROJECT_ROOT / CFG['reports']['metrics_dir']
METRICS.mkdir(parents=True, exist_ok=True)

with open(PROCESSED / 'split_meta.yaml') as f:
    SPLIT = yaml.safe_load(f)

FEATURE_COLS = SPLIT['feature_cols']
TARGET_COL   = SPLIT['target_col']
TS_COL       = SPLIT['ts_col']
TRAIN_END    = SPLIT['train_end']
VAL_END      = SPLIT['val_end']
TEST_END     = SPLIT['test_end']

print('Setup complete.')

---
## 1 · Consolidated Metrics Table

In [ ]:
# ── Load metrics collected by notebook 03 ────────────────────────────
with open(PROCESSED / 'metrics_draft.yaml') as f:
    results_raw = yaml.safe_load(f)

results_df = pd.DataFrame(results_raw)
results_df = results_df.sort_values(['partition','mae']).reset_index(drop=True)

# Save to metrics dir
metrics_csv = METRICS / 'forecast_metrics.csv'
results_df.to_csv(metrics_csv, index=False)
print(f'Metrics saved → {metrics_csv}')

# Display styled table
results_df.style.format({'mae':'{:.6f}','rmse':'{:.6f}','smape':'{:.2f}%'}) \
                .background_gradient(subset=['mae','rmse'], cmap='RdYlGn_r') \
                .set_caption('Model Evaluation — All Partitions')

In [ ]:
# ── Test-set comparison bar chart ────────────────────────────────────
test_res = results_df[results_df['partition'] == 'Test']

fig = go.Figure()
for metric, col in [('mae','#4FC3F7'), ('rmse','#F7931A')]:
    fig.add_trace(go.Bar(x=test_res['model'], y=test_res[metric],
                         name=metric.upper(), marker_color=col))
fig.update_layout(barmode='group', title='Test-Set Metrics Comparison',
                  template='plotly_dark', yaxis_title='Error')
fig.write_html(str(FIGURES / 'metrics_comparison.html'), include_plotlyjs='cdn')


In [ ]:
# ── sMAPE comparison ─────────────────────────────────────────────────
fig = go.Figure(go.Bar(
    x=test_res['model'], y=test_res['smape'],
    marker_color='#66BB6A', name='sMAPE %',
))
fig.update_layout(title='Test-Set sMAPE Comparison',
                  template='plotly_dark', yaxis_title='sMAPE (%)')
fig.write_html(str(FIGURES / 'smape_comparison.html'), include_plotlyjs='cdn')


---
## 2 · Forecast Diagnostic Plots

In [ ]:
# Residual analysis for the best available tree model

tree_artifacts = {"LightGBM": "lgbm_return.pkl", "CatBoost": "catboost_return.pkl"}
tree_results = test_res[test_res["model"].isin(tree_artifacts)].sort_values("mae")
if tree_results.empty:
    raise ValueError("No LightGBM or CatBoost test result is available for residual analysis.")
best = tree_results.iloc[0]
test_df = pd.read_parquet(PROCESSED / "test.parquet")
test_df[TS_COL] = pd.to_datetime(test_df[TS_COL], utc=True)
test_df = test_df.sort_values(TS_COL).reset_index(drop=True)
best_model_path = MODELS / tree_artifacts[best["model"]]
if not best_model_path.is_file():
    raise FileNotFoundError(f"Missing model artifact: {best_model_path}. Re-run notebook 03.")
artifact = joblib.load(best_model_path)
X_test = test_df[FEATURE_COLS]
y_test = test_df[TARGET_COL].to_numpy()
predictions = artifact["model"].predict(artifact["scaler"].transform(X_test))
if len(y_test) != len(predictions):
    raise ValueError("Test actuals and predictions have different lengths.")
residuals = y_test - predictions
print(f"Residual model: {best['model']} (test MAE {best['mae']:.6f})")

fig = go.Figure([
    go.Scatter(x=test_df[TS_COL], y=residuals, mode="lines", name="Actual ? predicted"),
    go.Scatter(x=test_df[TS_COL], y=[0] * len(residuals), name="Zero", line=dict(color="#aaa", dash="dot")),
])
fig.update_layout(title=f"{best['model']} ? Test Residuals", template="plotly_dark")
fig.write_html(str(FIGURES / "best_tree_residuals.html"), include_plotlyjs="cdn")
fig2 = go.Figure(go.Histogram(x=residuals, nbinsx=100, histnorm="probability density"))
fig2.update_layout(title=f"{best['model']} ? Residual Distribution", template="plotly_dark")
fig2.write_html(str(FIGURES / "best_tree_residual_dist.html"), include_plotlyjs="cdn")
print("Saved best-tree residual figures.")

---
## 3 · Artifact Export

In [ ]:
# ── Verify processed feature Parquet exists (created by notebook 02) ─
features_path = PROCESSED / 'btc_features.parquet'
if features_path.exists():
    df_feat = pd.read_parquet(features_path)
    print(f'btc_features.parquet : {df_feat.shape}  ✅')
else:
    print('⚠️  btc_features.parquet not found — re-run notebook 02.')

In [ ]:
# ── Log all runs to configs/experiments.yaml ──────────────────────────
for row in results_raw:
    if row['partition'] == 'Test':
        artifact_p = str(MODELS / f'{row["model"].lower()}_return.pkl')
        log_run(
            model         = row['model'],
            target        = TARGET_COL,
            train_end     = TRAIN_END,
            val_end       = VAL_END,
            test_end      = TEST_END,
            metrics       = {k: row[k] for k in ('mae','rmse','smape')},
            artifact_path = artifact_p if Path(artifact_p).exists() else 'n/a',
        )
print('All runs logged to configs/experiments.yaml.')

---
## 4 · Dashboard Readiness Check

In [ ]:
# Dashboard artifact readiness

required_for_dashboard = [PROCESSED / "btc_features.parquet"]
optional_for_dashboard = [
    MODELS / "lgbm_return.pkl", MODELS / "catboost_return.pkl",
    MODELS / "gmm.pkl", MODELS / "hmm.pkl",
    MODELS / "isolation_forest.pkl", MODELS / "anomaly_scores.csv",
    MODELS / "changepoints.csv", MODELS / "garch_cond_vol.csv",
    METRICS / "forecast_metrics.csv",
]
missing = [path for path in required_for_dashboard if not path.exists()]
print("Required:")
for path in required_for_dashboard:
    print(f"{'READY' if path.exists() else 'MISSING'}  {path.relative_to(PROJECT_ROOT)}")
print("Optional:")
for path in optional_for_dashboard:
    print(f"{'available' if path.exists() else 'not generated'}  {path.relative_to(PROJECT_ROOT)}")
if missing:
    raise FileNotFoundError(f"Missing required dashboard data: {missing}")
print("Launch the dashboard with: streamlit run Home.py")

In [ ]:
# ── Final summary ─────────────────────────────────────────────────────
print('=' * 60)
print('  PIPELINE COMPLETE')
print('=' * 60)
print(f'  Processed data : {PROCESSED}')
print(f'  Model artifacts: {MODELS}')
print(f'  Figures        : {FIGURES}')
print(f'  Metrics        : {METRICS}')
print()
print('Best test-set results:')
best = results_df[results_df['partition']=='Test'].sort_values('mae').head(3)
print(best[['model','mae','rmse','smape']].to_string(index=False))